In [1]:
from pathlib import Path
from io import StringIO
import re
import time
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

In [2]:
# Project folders
BASE_DIR = Path.cwd().parent

RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Base folder:", BASE_DIR)
print("Raw data folder:", RAW_DIR)
print("Processed data folder:", PROCESSED_DIR)

Base folder: C:\Users\DELL\afl-attendance-analytics
Raw data folder: C:\Users\DELL\afl-attendance-analytics\data\raw
Processed data folder: C:\Users\DELL\afl-attendance-analytics\data\processed


In [3]:
# AFL Tables attendance summary page
SUMMARY_URL = "https://afltables.com/afl/crowds/summary.html"

SUMMARY_URL

'https://afltables.com/afl/crowds/summary.html'

In [4]:
response = requests.get(
    SUMMARY_URL,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", response.status_code)

Status code: 200


In [5]:
html = response.text

raw_summary_path = RAW_DIR / "afl_attendance_summary.html"
raw_summary_path.write_text(html, encoding="utf-8")

print("Saved raw HTML file to:", raw_summary_path)

Saved raw HTML file to: C:\Users\DELL\afl-attendance-analytics\data\raw\afl_attendance_summary.html


In [6]:
tables = pd.read_html(StringIO(html))

print("Number of tables found:", len(tables))

Number of tables found: 14


In [7]:
tables[0].head()

Attendances by Season                                                        \
     Unnamed: 0_level_1 Home & Away                        Finals               
                   Year  Attendance   GM   Ave.    +/- Attendance   GM   Ave.   
0                  2026     3471068   90  38567   +4.3        NaN  NaN    NaN   
1                  2025     7656090  207  36986   -1.3     600937    9  66771   
2                  2024     7753251  207  37455   +3.7     533520    9  59280   
3                  2023     7474684  207  36110  +17.0     664780    9  73864   
4                  2022     6112431  198  30871  +30.5     639980    9  71109   

                                         
             Overall                     
      +/- Attendance   GM   Ave.    +/-  
0     NaN    3471068   90  38567   +0.9  
1   +12.6    8257027  216  38227   -0.4  
2   -19.7    8286771  216  38365   +1.8  
3    +3.9    8139464  216  37683  +15.5  
4  +134.6    6752411  207  32620  +35.9

In [8]:
season_raw = tables[0].copy()

print("Shape:", season_raw.shape)
season_raw.head()

Shape: (109, 13)


Attendances by Season                                                        \
     Unnamed: 0_level_1 Home & Away                        Finals               
                   Year  Attendance   GM   Ave.    +/- Attendance   GM   Ave.   
0                  2026     3471068   90  38567   +4.3        NaN  NaN    NaN   
1                  2025     7656090  207  36986   -1.3     600937    9  66771   
2                  2024     7753251  207  37455   +3.7     533520    9  59280   
3                  2023     7474684  207  36110  +17.0     664780    9  73864   
4                  2022     6112431  198  30871  +30.5     639980    9  71109   

                                         
             Overall                     
      +/- Attendance   GM   Ave.    +/-  
0     NaN    3471068   90  38567   +0.9  
1   +12.6    8257027  216  38227   -0.4  
2   -19.7    8286771  216  38365   +1.8  
3    +3.9    8139464  216  37683  +15.5  
4  +134.6    6752411  207  32620  +35.9

In [9]:
# I am checking how pandas stored the column names before cleaning them.

season_raw.columns

MultiIndex([('Attendances by Season', 'Unnamed: 0_level_1',       'Year'),
            ('Attendances by Season',        'Home & Away', 'Attendance'),
            ('Attendances by Season',        'Home & Away',         'GM'),
            ('Attendances by Season',        'Home & Away',       'Ave.'),
            ('Attendances by Season',        'Home & Away',        '+/-'),
            ('Attendances by Season',             'Finals', 'Attendance'),
            ('Attendances by Season',             'Finals',         'GM'),
            ('Attendances by Season',             'Finals',       'Ave.'),
            ('Attendances by Season',             'Finals',        '+/-'),
            ('Attendances by Season',            'Overall', 'Attendance'),
            ('Attendances by Season',            'Overall',         'GM'),
            ('Attendances by Season',            'Overall',       'Ave.'),
            ('Attendances by Season',            'Overall',        '+/-')],
           )

In [10]:
# I am creating a clean copy before renaming columns so the original raw table stays unchanged.

season_clean = season_raw.copy()

season_clean.columns = [
    "season",
    "home_away_attendance",
    "home_away_games",
    "home_away_average",
    "home_away_change",
    "finals_attendance",
    "finals_games",
    "finals_average",
    "finals_change",
    "overall_attendance",
    "overall_games",
    "overall_average",
    "overall_change"
]

season_clean.head()

,season,home_away_attendance,home_away_games,home_away_average,home_away_change,finals_attendance,finals_games,finals_average,finals_change,overall_attendance,overall_games,overall_average,overall_change
0,2026,3471068,90,38567,+4.3,NaN,NaN,NaN,NaN,3471068,90,38567,+0.9
1,2025,7656090,207,36986,-1.3,600937,9,66771,+12.6,8257027,216,38227,-0.4
2,2024,7753251,207,37455,+3.7,533520,9,59280,-19.7,8286771,216,38365,+1.8
3,2023,7474684,207,36110,+17.0,664780,9,73864,+3.9,8139464,216,37683,+15.5
4,2022,6112431,198,30871,+30.5,639980,9,71109,+134.6,6752411,207,32620,+35.9


In [11]:
# I am checking the data types to see whether numeric columns are stored as numbers or text.

season_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   season                109 non-null    object
 1   home_away_attendance  109 non-null    object
 2   home_away_games       109 non-null    object
 3   home_away_average     109 non-null    object
 4   home_away_change      106 non-null    object
 5   finals_attendance     108 non-null    object
 6   finals_games          108 non-null    object
 7   finals_average        108 non-null    object
 8   finals_change         105 non-null    object
 9   overall_attendance    109 non-null    object
 10  overall_games         109 non-null    object
 11  overall_average       109 non-null    object
 12  overall_change        106 non-null    object
dtypes: object(13)
memory usage: 11.2+ KB


In [12]:
# I am inspecting sample values before converting columns to numeric types.

season_clean[
    [
        "season",
        "home_away_attendance",
        "home_away_games",
        "home_away_average",
        "home_away_change",
        "finals_attendance",
        "overall_attendance",
        "overall_change"
    ]
].head(10)

,season,home_away_attendance,home_away_games,home_away_average,home_away_change,finals_attendance,overall_attendance,overall_change
0,2026,3471068,90,38567,+4.3,NaN,3471068,+0.9
1,2025,7656090,207,36986,-1.3,600937,8257027,-0.4
2,2024,7753251,207,37455,+3.7,533520,8286771,+1.8
3,2023,7474684,207,36110,+17.0,664780,8139464,+15.5
4,2022,6112431,198,30871,+30.5,639980,6752411,+35.9
5,2021*,3809275,161,23660,+255.0,272746,4082021,+209.1
6,2020*,826458,124,6665,-81.0,206579,1033037,-78.6
7,2019,6954187,198,35122,+0.9,563460,7517647,-1.0
8,2018,6893909,198,34818,+2.4,700393,7594302,+4.2
9,2017,6733960,198,34010,+6.8,553818,7287778,+6.2


In [13]:
# I am extracting the four-digit season year first.
# Rows that do not contain a year are summary rows, not season records.

season_year_check = season_clean["season"].astype(str).str.extract(r"(\d{4})", expand=False)

season_clean.loc[season_year_check.isna(), ["season"]]

,season
106,Totals
107,Excl 20/21
108,* excludes matches played behind closed doors


In [14]:
# I am keeping only rows that contain a valid four-digit season year.
# This removes summary rows such as Totals and Excl 20/21.

season_clean = season_clean.loc[season_year_check.notna()].copy()

season_clean["season"] = season_clean["season"].astype(str).str.extract(r"(\d{4})", expand=False).astype(int)

season_clean[["season"]].head(10)

,season
0,2026
1,2025
2,2024
3,2023
4,2022
5,2021
6,2020
7,2019
8,2018
9,2017


In [15]:
# I am checking the cleaned season column and confirming that non-season rows were removed.

print("Shape after removing non-season rows:", season_clean.shape)

print("\nFirst 10 seasons:")
print(season_clean["season"].head(10))

print("\nLast 10 seasons:")
print(season_clean["season"].tail(10))

print("\nData type of season column:")
print(season_clean["season"].dtype)

Shape after removing non-season rows: (106, 13)

First 10 seasons:
0    2026
1    2025
2    2024
3    2023
4    2022
5    2021
6    2020
7    2019
8    2018
9    2017
Name: season, dtype: int64

Last 10 seasons:
96     1930
97     1929
98     1928
99     1927
100    1926
101    1925
102    1924
103    1923
104    1922
105    1921
Name: season, dtype: int64

Data type of season column:
int64


In [16]:
# I am converting the attendance, games, average and change columns into numeric values.
# This is needed so Python and SQL can calculate averages, sums, rankings and trends correctly.

numeric_columns = [
    "home_away_attendance",
    "home_away_games",
    "home_away_average",
    "home_away_change",
    "finals_attendance",
    "finals_games",
    "finals_average",
    "finals_change",
    "overall_attendance",
    "overall_games",
    "overall_average",
    "overall_change"
]

for column in numeric_columns:
    season_clean[column] = (
        season_clean[column]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("+", "", regex=False)
        .str.strip()
    )
    
    season_clean[column] = pd.to_numeric(season_clean[column], errors="coerce")

season_clean.head(10)

,season,home_away_attendance,home_away_games,home_away_average,home_away_change,finals_attendance,finals_games,finals_average,finals_change,overall_attendance,overall_games,overall_average,overall_change
0,2026,3471068,90,38567,4.3,NaN,NaN,NaN,NaN,3471068,90,38567,0.9
1,2025,7656090,207,36986,-1.3,600937.0,9.0,66771.0,12.6,8257027,216,38227,-0.4
2,2024,7753251,207,37455,3.7,533520.0,9.0,59280.0,-19.7,8286771,216,38365,1.8
3,2023,7474684,207,36110,17.0,664780.0,9.0,73864.0,3.9,8139464,216,37683,15.5
4,2022,6112431,198,30871,30.5,639980.0,9.0,71109.0,134.6,6752411,207,32620,35.9
5,2021,3809275,161,23660,255.0,272746.0,9.0,30305.0,32.0,4082021,170,24012,209.1
6,2020,826458,124,6665,-81.0,206579.0,9.0,22953.0,-63.3,1033037,133,7767,-78.6
7,2019,6954187,198,35122,0.9,563460.0,9.0,62607.0,-19.6,7517647,207,36317,-1.0
8,2018,6893909,198,34818,2.4,700393.0,9.0,77821.0,26.5,7594302,207,36687,4.2
9,2017,6733960,198,34010,6.8,553818.0,9.0,61535.0,-0.8,7287778,207,35207,6.2


In [17]:
# I am verifying the data types.

season_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 106 entries, 0 to 105
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   season                106 non-null    int64  
 1   home_away_attendance  106 non-null    int64  
 2   home_away_games       106 non-null    int64  
 3   home_away_average     106 non-null    int64  
 4   home_away_change      105 non-null    float64
 5   finals_attendance     105 non-null    float64
 6   finals_games          105 non-null    float64
 7   finals_average        105 non-null    float64
 8   finals_change         104 non-null    float64
 9   overall_attendance    106 non-null    int64  
 10  overall_games         106 non-null    int64  
 11  overall_average       106 non-null    int64  
 12  overall_change        105 non-null    float64
dtypes: float64(6), int64(7)
memory usage: 11.6 KB


In [18]:
# I am adding flags to separate the full historical dataset from the main analysis period.
# This lets me keep all seasons in the clean dataset while filtering the main analysis clearly.

MAIN_START_SEASON = 2012
MAIN_END_SEASON = 2025
COVID_AFFECTED_SEASONS = [2020, 2021]

season_clean["covid_affected"] = season_clean["season"].isin(COVID_AFFECTED_SEASONS)

season_clean["included_in_main_analysis"] = (
    (season_clean["season"] >= MAIN_START_SEASON)
    & (season_clean["season"] <= MAIN_END_SEASON)
    & (~season_clean["season"].isin(COVID_AFFECTED_SEASONS))
)

season_clean[
    [
        "season",
        "home_away_attendance",
        "home_away_average",
        "overall_attendance",
        "overall_average",
        "covid_affected",
        "included_in_main_analysis"
    ]
].head(15)

,season,home_away_attendance,home_away_average,overall_attendance,overall_average,covid_affected,included_in_main_analysis
0,2026,3471068,38567,3471068,38567,False,False
1,2025,7656090,36986,8257027,38227,False,True
2,2024,7753251,37455,8286771,38365,False,True
3,2023,7474684,36110,8139464,37683,False,True
4,2022,6112431,30871,6752411,32620,False,True
5,2021,3809275,23660,4082021,24012,True,False
6,2020,826458,6665,1033037,7767,True,False
7,2019,6954187,35122,7517647,36317,False,True
8,2018,6893909,34818,7594302,36687,False,True
9,2017,6733960,34010,7287778,35207,False,True


In [19]:
# I am checking whether the COVID and main analysis flags were created correctly.

print("COVID affected season counts:")
print(season_clean["covid_affected"].value_counts())

print("\nMain analysis inclusion counts:")
print(season_clean["included_in_main_analysis"].value_counts())

print("\nSeasons included in main analysis:")
season_clean.loc[
    season_clean["included_in_main_analysis"] == True,
    ["season", "home_away_average", "overall_average", "included_in_main_analysis"]
].sort_values("season")

COVID affected season counts:
covid_affected
False    104
True       2
Name: count, dtype: int64

Main analysis inclusion counts:
included_in_main_analysis
False    94
True     12
Name: count, dtype: int64

Seasons included in main analysis:


,season,home_away_average,overall_average,included_in_main_analysis
14,2012,31509,32748,True
13,2013,32163,33461,True
12,2014,32333,33680,True
11,2015,32257,33367,True
10,2016,31850,33163,True
9,2017,34010,35207,True
8,2018,34818,36687,True
7,2019,35122,36317,True
4,2022,30871,32620,True
3,2023,36110,37683,True


In [20]:
# I am sorting the season-level dataset from oldest to newest.
# Resetting the index gives the cleaned dataset a fresh row order after filtering and sorting.

season_clean = season_clean.sort_values("season").reset_index(drop=True)

season_clean.head()

,season,home_away_attendance,home_away_games,home_away_average,home_away_change,finals_attendance,finals_games,finals_average,finals_change,overall_attendance,overall_games,overall_average,overall_change,covid_affected,included_in_main_analysis
0,1921,1175408,72,16325,NaN,165450.0,4.0,41363.0,NaN,1340858,76,17643,NaN,False,False
1,1922,1324792,72,18400,12.7,207278.0,4.0,51820.0,25.3,1532070,76,20159,14.3,False,False
2,1923,1331426,72,18492,0.5,213462.0,4.0,53366.0,3.0,1544888,76,20327,0.8,False,False
3,1924,1475541,72,20494,10.8,170732.0,6.0,28455.0,-46.7,1646273,78,21106,3.8,False,False
4,1925,1645904,102,16136,-21.3,225432.0,4.0,56358.0,98.1,1871336,106,17654,-16.4,False,False


In [21]:
# I am saving the cleaned season-level attendance dataset as a CSV file.
# This processed file will be used later for SQL analysis and visualisation.

season_output_path = PROCESSED_DIR / "afl_season_attendance_clean.csv"

season_clean.to_csv(season_output_path, index=False)

print("Cleaned season-level dataset saved to:")
print(season_output_path)

Cleaned season-level dataset saved to:
C:\Users\DELL\afl-attendance-analytics\data\processed\afl_season_attendance_clean.csv


In [22]:
# I am reloading the saved CSV to confirm the processed file was saved correctly.

season_check = pd.read_csv(PROCESSED_DIR / "afl_season_attendance_clean.csv")

season_check.head()

,season,home_away_attendance,home_away_games,home_away_average,home_away_change,finals_attendance,finals_games,finals_average,finals_change,overall_attendance,overall_games,overall_average,overall_change,covid_affected,included_in_main_analysis
0,1921,1175408,72,16325,NaN,165450.0,4.0,41363.0,NaN,1340858,76,17643,NaN,False,False
1,1922,1324792,72,18400,12.7,207278.0,4.0,51820.0,25.3,1532070,76,20159,14.3,False,False
2,1923,1331426,72,18492,0.5,213462.0,4.0,53366.0,3.0,1544888,76,20327,0.8,False,False
3,1924,1475541,72,20494,10.8,170732.0,6.0,28455.0,-46.7,1646273,78,21106,3.8,False,False
4,1925,1645904,102,16136,-21.3,225432.0,4.0,56358.0,98.1,1871336,106,17654,-16.4,False,False


In [23]:
# I am confirming the final saved season-level dataset size.

print("Saved season dataset shape:", season_check.shape)
print("First season:", season_check["season"].min())
print("Latest season:", season_check["season"].max())
print("Main analysis seasons:", season_check["included_in_main_analysis"].sum())

Saved season dataset shape: (106, 15)
First season: 1921
Latest season: 2026
Main analysis seasons: 12


In [24]:
# I am creating a reusable AFL Tables season URL pattern.
# I am testing it first with the 2025 season before collecting all seasons.

SEASON_URL_TEMPLATE = "https://afltables.com/afl/seas/{season}.html"

test_season = 2025
test_season_url = SEASON_URL_TEMPLATE.format(season=test_season)

test_season_url

'https://afltables.com/afl/seas/2025.html'

In [25]:
# I am downloading the 2025 season page to inspect its match-level structure.

season_response = requests.get(
    test_season_url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30
)

print("Status code:", season_response.status_code)

Status code: 200


In [26]:
# I am saving the raw 2025 season HTML page before extracting any data.

season_html = season_response.text

raw_season_path = RAW_DIR / f"afl_season_{test_season}.html"
raw_season_path.write_text(season_html, encoding="utf-8")

print("Saved raw season HTML to:")
print(raw_season_path)

Saved raw season HTML to:
C:\Users\DELL\afl-attendance-analytics\data\raw\afl_season_2025.html


In [27]:
# I am reading all HTML tables from the 2025 AFL season page.

season_tables = pd.read_html(StringIO(season_html))

print("Number of tables found:", len(season_tables))

Number of tables found: 338


In [28]:
# I am checking the shape of each table to identify which one likely contains match-level data.

for i, table in enumerate(season_tables):
    print(f"Table {i}: shape = {table.shape}")

Table 0: shape = (1, 2)
Table 1: shape = (1, 2)
Table 2: shape = (2, 4)
Table 3: shape = (2, 4)
Table 4: shape = (2, 4)
Table 5: shape = (2, 4)
Table 6: shape = (1, 4)
Table 7: shape = (1, 4)
Table 8: shape = (1, 4)
Table 9: shape = (1, 4)
Table 10: shape = (1, 4)
Table 11: shape = (1, 4)
Table 12: shape = (1, 4)
Table 13: shape = (1, 4)
Table 14: shape = (1, 4)
Table 15: shape = (1, 4)
Table 16: shape = (5, 4)
Table 17: shape = (1, 2)
Table 18: shape = (1, 2)
Table 19: shape = (2, 4)
Table 20: shape = (2, 4)
Table 21: shape = (2, 4)
Table 22: shape = (2, 4)
Table 23: shape = (2, 4)
Table 24: shape = (2, 4)
Table 25: shape = (2, 4)
Table 26: shape = (2, 4)
Table 27: shape = (2, 4)
Table 28: shape = (19, 4)
Table 29: shape = (1, 2)
Table 30: shape = (1, 2)
Table 31: shape = (2, 4)
Table 32: shape = (2, 4)
Table 33: shape = (2, 4)
Table 34: shape = (2, 4)
Table 35: shape = (2, 4)
Table 36: shape = (2, 4)
Table 37: shape = (2, 4)
Table 38: shape = (2, 4)
Table 39: shape = (1, 4)
Table 40:

In [29]:
# I am previewing the first few tables to find the one that contains match results, attendance and venue.

for i, table in enumerate(season_tables[:5]):
    print(f"\nTable {i}: shape = {table.shape}")
    display(table.head())


Table 0: shape = (1, 2)


,0,1
0,Round 1 * see notes,"Rnd Att: 104,292 (26,073) Tot Att: 104,292 (26..."



Table 1: shape = (1, 2)


,0,1
0,Sydney 4.3 6.4 11.5 11.10 76Fri 07-Mar-2025 7:...,Rd 1 Ladder GW 1 4200.0 HW 1 4126.3 SY 1 079.2...



Table 2: shape = (2, 4)


,0,1,2,3
0,Sydney,4.3 6.4 11.5 11.10,76,"Fri 07-Mar-2025 7:40 PM (6:40 PM) Att: 40,310 ..."
1,Hawthorn,5.3 9.6 11.10 14.12,96,Hawthorn won by 20 pts [Match stats]



Table 3: shape = (2, 4)


,0,1,2,3
0,Greater Western Sydney,5.3 8.7 9.9 15.14,104,"Sun 09-Mar-2025 3:20 PM (2:20 PM) Att: 19,248 ..."
1,Collingwood,1.6 4.8 4.13 6.16,52,Greater Western Sydney won by 52 pts [Match st...



Table 4: shape = (2, 4)


,0,1,2,3
0,Brisbane Lions,2.2 3.4 7.7 10.10,70,"Sat 29-Mar-2025 6:35 PM Att: 27,966 Venue: Gabba"
1,Geelong,3.4 7.6 7.6 9.7,61,Brisbane Lions won by 9 pts [Match stats]


In [30]:
# I am inspecting one match table to understand its exact structure before writing a parser.

sample_match_table = season_tables[2]

sample_match_table

,0,1,2,3
0,Sydney,4.3 6.4 11.5 11.10,76,"Fri 07-Mar-2025 7:40 PM (6:40 PM) Att: 40,310 ..."
1,Hawthorn,5.3 9.6 11.10 14.12,96,Hawthorn won by 20 pts [Match stats]


In [31]:
# I am printing the full values from the sample match table.
# The normal table view cuts long text with "...", so this helps me see the full match details.

print("Shape:", sample_match_table.shape)

print("\nRow 0 full values:")
for col in sample_match_table.columns:
    print(f"Column {col}: {repr(sample_match_table.loc[0, col])}")

print("\nRow 1 full values:")
for col in sample_match_table.columns:
    print(f"Column {col}: {repr(sample_match_table.loc[1, col])}")

Shape: (2, 4)

Row 0 full values:
Column 0: 'Sydney'
Column 1: '4.3 6.4 11.5 11.10'
Column 2: np.int64(76)
Column 3: 'Fri 07-Mar-2025 7:40 PM (6:40 PM) Att: 40,310 Venue: S.C.G.'

Row 1 full values:
Column 0: 'Hawthorn'
Column 1: '5.3 9.6 11.10 14.12'
Column 2: np.int64(96)
Column 3: 'Hawthorn won by 20 pts [Match stats]'


In [32]:
# I am manually extracting values from one sample match table first.
# This helps me confirm the logic before creating a reusable function.

row_0 = sample_match_table.iloc[0]
row_1 = sample_match_table.iloc[1]

team_1 = row_0[0]
team_2 = row_1[0]

team_1_score = row_0[2]
team_2_score = row_1[2]

match_details = row_0[3]
result_details = row_1[3]

print("Team 1:", team_1)
print("Team 2:", team_2)
print("Team 1 score:", team_1_score)
print("Team 2 score:", team_2_score)
print("Match details:", match_details)
print("Result details:", result_details)

Team 1: Sydney
Team 2: Hawthorn
Team 1 score: 76
Team 2 score: 96
Match details: Fri 07-Mar-2025 7:40 PM (6:40 PM) Att: 40,310 Venue: S.C.G.
Result details: Hawthorn won by 20 pts [Match stats]


In [33]:
# I am extracting match day, date, time, attendance and venue from the match details text.

match_pattern = re.search(
    r"(?P<match_day>[A-Za-z]{3})\s+"
    r"(?P<match_date>\d{2}-[A-Za-z]{3}-\d{4})\s+"
    r"(?P<match_time>\d{1,2}:\d{2}\s+[AP]M)"
    r"(?:\s+\([^)]+\))?\s+"
    r"Att:\s+(?P<attendance>[\d,]+)\s+"
    r"Venue:\s+(?P<venue>.+)",
    match_details
)

if match_pattern:
    print("Match day:", match_pattern.group("match_day"))
    print("Match date:", match_pattern.group("match_date"))
    print("Match time:", match_pattern.group("match_time"))
    print("Attendance:", match_pattern.group("attendance"))
    print("Venue:", match_pattern.group("venue"))
else:
    print("No match found")

Match day: Fri
Match date: 07-Mar-2025
Match time: 7:40 PM
Attendance: 40,310
Venue: S.C.G.


In [34]:
# I am extracting the winner and margin from the result details text.

result_pattern = re.search(
    r"(?P<winner>.+?)\s+won by\s+(?P<margin>\d+)\s+pts",
    result_details
)

if result_pattern:
    print("Winner:", result_pattern.group("winner"))
    print("Margin:", result_pattern.group("margin"))
else:
    print("No result pattern found")

Winner: Hawthorn
Margin: 20


In [35]:
# I am combining the extracted values from one match table into a clean dictionary.
# This dictionary represents one match record in the final match-level dataset.

single_match_record = {
    "season": test_season,
    "round": 1,
    "home_team": team_1,
    "away_team": team_2,
    "home_score": int(team_1_score),
    "away_score": int(team_2_score),
    "match_day": match_pattern.group("match_day"),
    "match_date": match_pattern.group("match_date"),
    "match_time": match_pattern.group("match_time"),
    "attendance": int(match_pattern.group("attendance").replace(",", "")),
    "venue": match_pattern.group("venue"),
    "winner": result_pattern.group("winner"),
    "margin": int(result_pattern.group("margin"))
}

single_match_record

{'season': 2025,
 'round': 1,
 'home_team': 'Sydney',
 'away_team': 'Hawthorn',
 'home_score': 76,
 'away_score': 96,
 'match_day': 'Fri',
 'match_date': '07-Mar-2025',
 'match_time': '7:40 PM',
 'attendance': 40310,
 'venue': 'S.C.G.',
 'winner': 'Hawthorn',
 'margin': 20}

In [36]:
# I am improving the parser so it can handle normal wins, 1-point wins and drawn matches.

def parse_match_table(match_table, season, round_number):
    row_0 = match_table.iloc[0]
    row_1 = match_table.iloc[1]

    team_1 = row_0[0]
    team_2 = row_1[0]

    team_1_score = row_0[2]
    team_2_score = row_1[2]

    match_details = str(row_0[3])
    result_details = str(row_1[3])

    match_pattern = re.search(
        r"(?P<match_day>[A-Za-z]{3})\s+"
        r"(?P<match_date>\d{2}-[A-Za-z]{3}-\d{4})\s+"
        r"(?P<match_time>\d{1,2}:\d{2}\s+[AP]M)"
        r"(?:\s+\([^)]+\))?\s+"
        r"Att:\s+(?P<attendance>[\d,]+)\s+"
        r"Venue:\s+(?P<venue>.+)",
        match_details
    )

    if not match_pattern:
        return None

    win_pattern = re.search(
        r"(?P<winner>.+?)\s+won by\s+(?P<margin>\d+)\s+pts?",
        result_details
    )

    draw_pattern = re.search(
        r"draw|drawn",
        result_details,
        re.IGNORECASE
    )

    if win_pattern:
        winner = win_pattern.group("winner")
        margin = int(win_pattern.group("margin"))
        result_type = "Win"
    elif draw_pattern:
        winner = "Draw"
        margin = 0
        result_type = "Draw"
    else:
        return None

    match_record = {
        "season": season,
        "round": round_number,
        "home_team": team_1,
        "away_team": team_2,
        "home_score": int(team_1_score),
        "away_score": int(team_2_score),
        "match_day": match_pattern.group("match_day"),
        "match_date": match_pattern.group("match_date"),
        "match_time": match_pattern.group("match_time"),
        "attendance": int(match_pattern.group("attendance").replace(",", "")),
        "venue": match_pattern.group("venue"),
        "winner": winner,
        "margin": margin,
        "result_type": result_type
    }

    return match_record

In [37]:
# I am testing the reusable parser on the sample match table.

test_match_record = parse_match_table(
    match_table=season_tables[2],
    season=2025,
    round_number=1
)

test_match_record

{'season': 2025,
 'round': 1,
 'home_team': 'Sydney',
 'away_team': 'Hawthorn',
 'home_score': 76,
 'away_score': 96,
 'match_day': 'Fri',
 'match_date': '07-Mar-2025',
 'match_time': '7:40 PM',
 'attendance': 40310,
 'venue': 'S.C.G.',
 'winner': 'Hawthorn',
 'margin': 20,
 'result_type': 'Win'}

In [38]:
# I am testing the parser across the first few tables from the 2025 season page.
# This helps me check which tables are match tables and which tables should be skipped.

test_records = []

for table_index, table in enumerate(season_tables[:20]):
    parsed_record = None
    
    if table.shape == (2, 4):
        parsed_record = parse_match_table(
            match_table=table,
            season=test_season,
            round_number=1
        )
    
    if parsed_record is not None:
        parsed_record["table_index"] = table_index
        test_records.append(parsed_record)

test_records_df = pd.DataFrame(test_records)

test_records_df

,season,round,home_team,away_team,home_score,away_score,match_day,match_date,match_time,attendance,venue,winner,margin,result_type,table_index
0,2025,1,Sydney,Hawthorn,76,96,Fri,07-Mar-2025,7:40 PM,40310,S.C.G.,Hawthorn,20,Win,2
1,2025,1,Greater Western Sydney,Collingwood,104,52,Sun,09-Mar-2025,3:20 PM,19248,Sydney Showground,Greater Western Sydney,52,Win,3
2,2025,1,Brisbane Lions,Geelong,70,61,Sat,29-Mar-2025,6:35 PM,27966,Gabba,Brisbane Lions,9,Win,4
3,2025,1,Gold Coast,Essendon,153,58,Wed,27-Aug-2025,7:20 PM,16768,Carrara,Gold Coast,95,Win,5
4,2025,1,Richmond,Carlton,82,69,Thu,13-Mar-2025,7:30 PM,80009,M.C.G.,Richmond,13,Win,19


In [39]:
# I am reparsing the 2025 season with proper match type labels.
# Home & Away matches keep their round number, while finals are labelled separately.

match_records_2025 = []
current_round = None
current_match_type = "Home & Away"
current_final_stage = None

for table_index, table in enumerate(season_tables):
    table_text = " ".join(table.astype(str).values.flatten()).strip()
    
    if table_text == "Finals":
        current_match_type = "Finals"
        current_round = None
        current_final_stage = None
        continue
    
    final_stage_match = re.search(
        r"(Qualifying Final|Elimination Final|Semi Final|Preliminary Final|Grand Final)",
        table_text
    )
    
    if final_stage_match:
        current_match_type = "Finals"
        current_final_stage = final_stage_match.group(1)
        continue
    
    round_match = re.search(r"\bRound\s+(\d+)\b", table_text)
    
    if round_match and current_match_type != "Finals":
        current_round = int(round_match.group(1))
        current_match_type = "Home & Away"
        current_final_stage = None
        continue
    
    if table.shape == (2, 4):
        parsed_record = parse_match_table(
            match_table=table,
            season=test_season,
            round_number=current_round
        )
        
        if parsed_record is not None:
            parsed_record["match_type"] = current_match_type
            parsed_record["final_stage"] = current_final_stage
            parsed_record["table_index"] = table_index
            match_records_2025.append(parsed_record)

match_2025_df = pd.DataFrame(match_records_2025)

print("Total parsed 2025 match records:", len(match_2025_df))

print("\nMatches by match type:")
print(match_2025_df["match_type"].value_counts())

print("\nFinal stages:")
print(match_2025_df["final_stage"].value_counts(dropna=False))

match_2025_df.tail(12)

Total parsed 2025 match records: 216

Matches by match type:
match_type
Home & Away    207
Finals           9
Name: count, dtype: int64

Final stages:
final_stage
None                 207
Qualifying Final       2
Elimination Final      2
Semi Final             2
Preliminary Final      2
Grand Final            1
Name: count, dtype: int64


,season,round,home_team,away_team,home_score,away_score,match_day,match_date,match_time,attendance,venue,winner,margin,result_type,match_type,final_stage,table_index
204,2025,25.0,Greater Western Sydney,St Kilda,104,93,Sun,24-Aug-2025,12:20 PM,10563,Sydney Showground,Greater Western Sydney,11,Win,Home & Away,None,314
205,2025,25.0,Western Bulldogs,Fremantle,97,112,Sun,24-Aug-2025,3:15 PM,44069,Docklands,Fremantle,15,Win,Home & Away,None,315
206,2025,25.0,Brisbane Lions,Hawthorn,89,79,Sun,24-Aug-2025,7:20 PM,32086,Gabba,Brisbane Lions,10,Win,Home & Away,None,316
207,2025,NaN,Adelaide,Collingwood,55,79,Thu,04-Sep-2025,7:10 PM,52187,Adelaide Oval,Collingwood,24,Win,Finals,Qualifying Final,321
208,2025,NaN,Geelong,Brisbane Lions,112,74,Fri,05-Sep-2025,7:40 PM,86364,M.C.G.,Geelong,38,Win,Finals,Qualifying Final,323
209,2025,NaN,Greater Western Sydney,Hawthorn,88,107,Sat,06-Sep-2025,3:15 PM,20634,Sydney Showground,Hawthorn,19,Win,Finals,Elimination Final,325
210,2025,NaN,Fremantle,Gold Coast,79,80,Sat,06-Sep-2025,5:35 PM,57507,Perth Stadium,Gold Coast,1,Win,Finals,Elimination Final,327
211,2025,NaN,Adelaide,Hawthorn,67,101,Fri,12-Sep-2025,7:10 PM,52005,Adelaide Oval,Hawthorn,34,Win,Finals,Semi Final,329
212,2025,NaN,Brisbane Lions,Gold Coast,100,47,Sat,13-Sep-2025,7:35 PM,36628,Gabba,Brisbane Lions,53,Win,Finals,Semi Final,331
213,2025,NaN,Geelong,Hawthorn,115,85,Fri,19-Sep-2025,7:40 PM,99567,M.C.G.,Geelong,30,Win,Finals,Preliminary Final,333


In [40]:
# I am cleaning the 2025 match-level dataset so dates, match type and round values are analysis-ready.

match_2025_clean = match_2025_df.copy()

match_2025_clean["match_date"] = pd.to_datetime(
    match_2025_clean["match_date"],
    format="%d-%b-%Y",
    errors="coerce"
)

match_2025_clean["month"] = match_2025_clean["match_date"].dt.month_name()
match_2025_clean["year"] = match_2025_clean["match_date"].dt.year

match_2025_clean.loc[
    match_2025_clean["match_type"] == "Finals",
    "round"
] = pd.NA

match_2025_clean.head()

,season,round,home_team,away_team,home_score,away_score,match_day,match_date,match_time,attendance,venue,winner,margin,result_type,match_type,final_stage,table_index,month,year
0,2025,1.0,Sydney,Hawthorn,76,96,Fri,2025-03-07,7:40 PM,40310,S.C.G.,Hawthorn,20,Win,Home & Away,None,2,March,2025
1,2025,1.0,Greater Western Sydney,Collingwood,104,52,Sun,2025-03-09,3:20 PM,19248,Sydney Showground,Greater Western Sydney,52,Win,Home & Away,None,3,March,2025
2,2025,1.0,Brisbane Lions,Geelong,70,61,Sat,2025-03-29,6:35 PM,27966,Gabba,Brisbane Lions,9,Win,Home & Away,None,4,March,2025
3,2025,1.0,Gold Coast,Essendon,153,58,Wed,2025-08-27,7:20 PM,16768,Carrara,Gold Coast,95,Win,Home & Away,None,5,August,2025
4,2025,2.0,Richmond,Carlton,82,69,Thu,2025-03-13,7:30 PM,80009,M.C.G.,Richmond,13,Win,Home & Away,None,19,March,2025


In [41]:
# I am checking the cleaned 2025 match dataset structure.

print("Shape:", match_2025_clean.shape)

print("\nMatch types:")
print(match_2025_clean["match_type"].value_counts())

print("\nFinal stages:")
print(match_2025_clean["final_stage"].value_counts(dropna=False))

print("\nData types:")
match_2025_clean.info()

Shape: (216, 19)

Match types:
match_type
Home & Away    207
Finals           9
Name: count, dtype: int64

Final stages:
final_stage
None                 207
Qualifying Final       2
Elimination Final      2
Semi Final             2
Preliminary Final      2
Grand Final            1
Name: count, dtype: int64

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   season       216 non-null    int64         
 1   round        207 non-null    float64       
 2   home_team    216 non-null    object        
 3   away_team    216 non-null    object        
 4   home_score   216 non-null    int64         
 5   away_score   216 non-null    int64         
 6   match_day    216 non-null    object        
 7   match_date   216 non-null    datetime64[ns]
 8   match_time   216 non-null    object        
 9   attendance   216 non-null    

In [42]:
# I am checking that finals are now separated properly from Home & Away rounds.

match_2025_clean.loc[
    match_2025_clean["match_type"] == "Finals",
    [
        "season",
        "round",
        "final_stage",
        "home_team",
        "away_team",
        "match_date",
        "attendance",
        "venue",
        "winner",
        "margin"
    ]
]

,season,round,final_stage,home_team,away_team,match_date,attendance,venue,winner,margin
207,2025,NaN,Qualifying Final,Adelaide,Collingwood,2025-09-04,52187,Adelaide Oval,Collingwood,24
208,2025,NaN,Qualifying Final,Geelong,Brisbane Lions,2025-09-05,86364,M.C.G.,Geelong,38
209,2025,NaN,Elimination Final,Greater Western Sydney,Hawthorn,2025-09-06,20634,Sydney Showground,Hawthorn,19
210,2025,NaN,Elimination Final,Fremantle,Gold Coast,2025-09-06,57507,Perth Stadium,Gold Coast,1
211,2025,NaN,Semi Final,Adelaide,Hawthorn,2025-09-12,52005,Adelaide Oval,Hawthorn,34
212,2025,NaN,Semi Final,Brisbane Lions,Gold Coast,2025-09-13,36628,Gabba,Brisbane Lions,53
213,2025,NaN,Preliminary Final,Geelong,Hawthorn,2025-09-19,99567,M.C.G.,Geelong,30
214,2025,NaN,Preliminary Final,Collingwood,Brisbane Lions,2025-09-20,96023,M.C.G.,Brisbane Lions,29
215,2025,NaN,Grand Final,Geelong,Brisbane Lions,2025-09-27,100022,M.C.G.,Brisbane Lions,47


In [43]:
# I am doing final clean-up on the 2025 match dataset before reusing the same logic for all seasons.

match_2025_clean = match_2025_clean.copy()

match_2025_clean["round"] = match_2025_clean["round"].astype("Int64")

match_2025_clean["final_stage"] = match_2025_clean["final_stage"].fillna("Not applicable")

match_2025_clean["is_final"] = match_2025_clean["match_type"] == "Finals"

match_2025_clean["included_in_main_analysis"] = match_2025_clean["match_type"] == "Home & Away"

match_2025_clean.head()

,season,round,home_team,away_team,home_score,away_score,match_day,match_date,match_time,attendance,...,winner,margin,result_type,match_type,final_stage,table_index,month,year,is_final,included_in_main_analysis
0,2025,1,Sydney,Hawthorn,76,96,Fri,2025-03-07,7:40 PM,40310,...,Hawthorn,20,Win,Home & Away,Not applicable,2,March,2025,False,True
1,2025,1,Greater Western Sydney,Collingwood,104,52,Sun,2025-03-09,3:20 PM,19248,...,Greater Western Sydney,52,Win,Home & Away,Not applicable,3,March,2025,False,True
2,2025,1,Brisbane Lions,Geelong,70,61,Sat,2025-03-29,6:35 PM,27966,...,Brisbane Lions,9,Win,Home & Away,Not applicable,4,March,2025,False,True
3,2025,1,Gold Coast,Essendon,153,58,Wed,2025-08-27,7:20 PM,16768,...,Gold Coast,95,Win,Home & Away,Not applicable,5,August,2025,False,True
4,2025,2,Richmond,Carlton,82,69,Thu,2025-03-13,7:30 PM,80009,...,Richmond,13,Win,Home & Away,Not applicable,19,March,2025,False,True


In [44]:
# I am validating the final cleaned 2025 match dataset.

print("Final 2025 dataset shape:", match_2025_clean.shape)

print("\nMatch type counts:")
print(match_2025_clean["match_type"].value_counts())

print("\nFinal stage counts:")
print(match_2025_clean["final_stage"].value_counts())

print("\nMain analysis inclusion:")
print(match_2025_clean["included_in_main_analysis"].value_counts())

print("\nData types:")
match_2025_clean.info()

Final 2025 dataset shape: (216, 21)

Match type counts:
match_type
Home & Away    207
Finals           9
Name: count, dtype: int64

Final stage counts:
final_stage
Not applicable       207
Qualifying Final       2
Elimination Final      2
Semi Final             2
Preliminary Final      2
Grand Final            1
Name: count, dtype: int64

Main analysis inclusion:
included_in_main_analysis
True     207
False      9
Name: count, dtype: int64

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216 entries, 0 to 215
Data columns (total 21 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   season                     216 non-null    int64         
 1   round                      207 non-null    Int64         
 2   home_team                  216 non-null    object        
 3   away_team                  216 non-null    object        
 4   home_score                 216 non-null    int64       

In [45]:
# I am creating a reusable function to parse all match tables from one AFL season.
# This avoids repeating the same parsing logic for every season.

def parse_season_tables(season_tables, season):
    match_records = []
    current_round = None
    current_match_type = "Home & Away"
    current_final_stage = None

    for table_index, table in enumerate(season_tables):
        table_text = " ".join(table.astype(str).values.flatten()).strip()

        if table_text == "Finals":
            current_match_type = "Finals"
            current_round = None
            current_final_stage = None
            continue

        final_stage_match = re.search(
            r"(Qualifying Final|Elimination Final|Semi Final|Preliminary Final|Grand Final)",
            table_text
        )

        if final_stage_match:
            current_match_type = "Finals"
            current_final_stage = final_stage_match.group(1)
            continue

        round_match = re.search(r"\bRound\s+(\d+)\b", table_text)

        if round_match and current_match_type != "Finals":
            current_round = int(round_match.group(1))
            current_match_type = "Home & Away"
            current_final_stage = None
            continue

        if table.shape == (2, 4):
            parsed_record = parse_match_table(
                match_table=table,
                season=season,
                round_number=current_round
            )

            if parsed_record is not None:
                parsed_record["match_type"] = current_match_type
                parsed_record["final_stage"] = current_final_stage
                parsed_record["table_index"] = table_index
                match_records.append(parsed_record)

    match_df = pd.DataFrame(match_records)

    if not match_df.empty:
        match_df["match_date"] = pd.to_datetime(
            match_df["match_date"],
            format="%d-%b-%Y",
            errors="coerce"
        )

        match_df["month"] = match_df["match_date"].dt.month_name()
        match_df["year"] = match_df["match_date"].dt.year

        match_df.loc[
            match_df["match_type"] == "Finals",
            "round"
        ] = pd.NA

        match_df["round"] = match_df["round"].astype("Int64")
        match_df["final_stage"] = match_df["final_stage"].fillna("Not applicable")
        match_df["is_final"] = match_df["match_type"] == "Finals"
        match_df["included_in_main_analysis"] = match_df["match_type"] == "Home & Away"

    return match_df

In [46]:
# I am testing the reusable season parser on the 2025 tables to confirm it gives the same result.

match_2025_test = parse_season_tables(
    season_tables=season_tables,
    season=2025
)

print("Shape:", match_2025_test.shape)

print("\nMatch type counts:")
print(match_2025_test["match_type"].value_counts())

print("\nFinal stage counts:")
print(match_2025_test["final_stage"].value_counts())

match_2025_test.tail(10)

Shape: (216, 21)

Match type counts:
match_type
Home & Away    207
Finals           9
Name: count, dtype: int64

Final stage counts:
final_stage
Not applicable       207
Qualifying Final       2
Elimination Final      2
Semi Final             2
Preliminary Final      2
Grand Final            1
Name: count, dtype: int64


,season,round,home_team,away_team,home_score,away_score,match_day,match_date,match_time,attendance,...,winner,margin,result_type,match_type,final_stage,table_index,month,year,is_final,included_in_main_analysis
206,2025,25,Brisbane Lions,Hawthorn,89,79,Sun,2025-08-24,7:20 PM,32086,...,Brisbane Lions,10,Win,Home & Away,Not applicable,316,August,2025,False,True
207,2025,<NA>,Adelaide,Collingwood,55,79,Thu,2025-09-04,7:10 PM,52187,...,Collingwood,24,Win,Finals,Qualifying Final,321,September,2025,True,False
208,2025,<NA>,Geelong,Brisbane Lions,112,74,Fri,2025-09-05,7:40 PM,86364,...,Geelong,38,Win,Finals,Qualifying Final,323,September,2025,True,False
209,2025,<NA>,Greater Western Sydney,Hawthorn,88,107,Sat,2025-09-06,3:15 PM,20634,...,Hawthorn,19,Win,Finals,Elimination Final,325,September,2025,True,False
210,2025,<NA>,Fremantle,Gold Coast,79,80,Sat,2025-09-06,5:35 PM,57507,...,Gold Coast,1,Win,Finals,Elimination Final,327,September,2025,True,False
211,2025,<NA>,Adelaide,Hawthorn,67,101,Fri,2025-09-12,7:10 PM,52005,...,Hawthorn,34,Win,Finals,Semi Final,329,September,2025,True,False
212,2025,<NA>,Brisbane Lions,Gold Coast,100,47,Sat,2025-09-13,7:35 PM,36628,...,Brisbane Lions,53,Win,Finals,Semi Final,331,September,2025,True,False
213,2025,<NA>,Geelong,Hawthorn,115,85,Fri,2025-09-19,7:40 PM,99567,...,Geelong,30,Win,Finals,Preliminary Final,333,September,2025,True,False
214,2025,<NA>,Collingwood,Brisbane Lions,71,100,Sat,2025-09-20,5:15 PM,96023,...,Brisbane Lions,29,Win,Finals,Preliminary Final,335,September,2025,True,False
215,2025,<NA>,Geelong,Brisbane Lions,75,122,Sat,2025-09-27,2:30 PM,100022,...,Brisbane Lions,47,Win,Finals,Grand Final,337,September,2025,True,False


In [47]:
# I am creating a function to download, save, read and parse one AFL season page.

def collect_season_match_data(season):
    season_url = SEASON_URL_TEMPLATE.format(season=season)
    
    response = requests.get(
        season_url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )
    
    response.raise_for_status()
    
    season_html = response.text
    
    raw_season_path = RAW_DIR / f"afl_season_{season}.html"
    raw_season_path.write_text(season_html, encoding="utf-8")
    
    season_tables = pd.read_html(StringIO(season_html))
    
    season_match_df = parse_season_tables(
        season_tables=season_tables,
        season=season
    )
    
    return season_match_df

In [48]:
# I am testing the full collection function on the 2024 season before collecting all seasons.

match_2024_test = collect_season_match_data(2024)

print("2024 shape:", match_2024_test.shape)

print("\nMatch type counts:")
print(match_2024_test["match_type"].value_counts())

print("\nFinal stage counts:")
print(match_2024_test["final_stage"].value_counts())

match_2024_test.head()

2024 shape: (216, 21)

Match type counts:
match_type
Home & Away    207
Finals           9
Name: count, dtype: int64

Final stage counts:
final_stage
Not applicable       207
Qualifying Final       2
Elimination Final      2
Semi Final             2
Preliminary Final      2
Grand Final            1
Name: count, dtype: int64


,season,round,home_team,away_team,home_score,away_score,match_day,match_date,match_time,attendance,...,winner,margin,result_type,match_type,final_stage,table_index,month,year,is_final,included_in_main_analysis
0,2024,1,Sydney,Melbourne,86,64,Thu,2024-03-07,7:30 PM,40012,...,Sydney,22,Win,Home & Away,Not applicable,2,March,2024,False,True
1,2024,1,Brisbane Lions,Carlton,85,86,Fri,2024-03-08,6:40 PM,33367,...,Carlton,1,Win,Home & Away,Not applicable,3,March,2024,False,True
2,2024,1,Gold Coast,Richmond,99,60,Sat,2024-03-09,3:20 PM,22086,...,Gold Coast,39,Win,Home & Away,Not applicable,4,March,2024,False,True
3,2024,1,Greater Western Sydney,Collingwood,114,82,Sat,2024-03-09,7:30 PM,21235,...,Greater Western Sydney,32,Win,Home & Away,Not applicable,5,March,2024,False,True
4,2024,2,Carlton,Richmond,86,81,Thu,2024-03-14,7:30 PM,83881,...,Carlton,5,Win,Home & Away,Not applicable,19,March,2024,False,True


In [49]:
# I am comparing the parsed 2024 match counts against the cleaned season-level summary.

season_2024_summary = season_clean.loc[
    season_clean["season"] == 2024,
    ["home_away_games", "finals_games", "overall_games"]
]

print("Season summary for 2024:")
display(season_2024_summary)

print("\nParsed match counts for 2024:")
print("Home & Away:", (match_2024_test["match_type"] == "Home & Away").sum())
print("Finals:", (match_2024_test["match_type"] == "Finals").sum())
print("Overall:", len(match_2024_test))

Season summary for 2024:


,home_away_games,finals_games,overall_games
103,207,9.0,216



Parsed match counts for 2024:
Home & Away: 207
Finals: 9
Overall: 216


In [50]:
# I am collecting match-level AFL attendance data for the modern AFL period.
# I am using 2012 onwards because this represents the current 18-team AFL structure.

START_SEASON = 2012
END_SEASON = 2025

all_match_datasets = []

for season in range(START_SEASON, END_SEASON + 1):
    print(f"Collecting season {season}...")
    
    season_match_df = collect_season_match_data(season)
    
    print(
        f"Season {season}: {len(season_match_df)} matches collected "
        f"({season_match_df['match_type'].value_counts().to_dict()})"
    )
    
    all_match_datasets.append(season_match_df)
    
    time.sleep(0.5)

match_all_clean = pd.concat(all_match_datasets, ignore_index=True)

print("\nAll seasons collected.")
print("Final shape:", match_all_clean.shape)

match_all_clean.head()

Season 2012: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2013: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2014: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2015: 206 matches collected ({'Home & Away': 197, 'Finals': 9})
Season 2016: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2017: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2018: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2019: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2020: 133 matches collected ({'Home & Away': 124, 'Finals': 9})
Season 2021: 170 matches collected ({'Home & Away': 161, 'Finals': 9})
Season 2022: 207 matches collected ({'Home & Away': 198, 'Finals': 9})
Season 2023: 216 matches collected ({'Home & Away': 207, 'Finals': 9})
Season 2024: 216 matches collected ({'Home & Away': 207, 'Finals': 9})
Season 2025: 216 matches collected ({'Home & Away': 207, 'Finals': 9})

All s

,season,round,home_team,away_team,home_score,away_score,match_day,match_date,match_time,attendance,...,winner,margin,result_type,match_type,final_stage,table_index,month,year,is_final,included_in_main_analysis
0,2012,1,Greater Western Sydney,Sydney,37,100,Sat,2012-03-24,7:20 PM,38203,...,Sydney,63,Win,Home & Away,Not applicable,2,March,2012,False,True
1,2012,1,Richmond,Carlton,81,125,Thu,2012-03-29,7:45 PM,78285,...,Carlton,44,Win,Home & Away,Not applicable,3,March,2012,False,True
2,2012,1,Hawthorn,Collingwood,137,115,Fri,2012-03-30,7:50 PM,78466,...,Hawthorn,22,Win,Home & Away,Not applicable,4,March,2012,False,True
3,2012,1,Melbourne,Brisbane Lions,78,119,Sat,2012-03-31,1:45 PM,33473,...,Brisbane Lions,41,Win,Home & Away,Not applicable,5,March,2012,False,True
4,2012,1,Gold Coast,Adelaide,68,137,Sat,2012-03-31,3:45 PM,12790,...,Adelaide,69,Win,Home & Away,Not applicable,6,March,2012,False,True


In [51]:
# I am checking the combined match-level dataset after collecting all seasons.

print("Combined match dataset shape:", match_all_clean.shape)

print("\nSeasons collected:")
print(sorted(match_all_clean["season"].unique()))

print("\nMatch type counts:")
print(match_all_clean["match_type"].value_counts())

print("\nRows by season and match type:")
match_all_clean.groupby(["season", "match_type"]).size().unstack(fill_value=0)

Combined match dataset shape: (2813, 21)

Seasons collected:
[np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Match type counts:
match_type
Home & Away    2687
Finals          126
Name: count, dtype: int64

Rows by season and match type:


match_type,Finals,Home & Away
season,,
2012,9,198
2013,9,198
2014,9,198
2015,9,197
2016,9,198
2017,9,198
2018,9,198
2019,9,198
2020,9,124


In [52]:
# I am comparing parsed match counts with the season-level summary to validate data quality.

parsed_counts = (
    match_all_clean
    .groupby(["season", "match_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

parsed_counts = parsed_counts.rename(columns={
    "Home & Away": "parsed_home_away_games",
    "Finals": "parsed_finals_games"
})

parsed_counts["parsed_overall_games"] = (
    parsed_counts["parsed_home_away_games"] + parsed_counts["parsed_finals_games"]
)

season_summary_counts = season_clean.loc[
    (season_clean["season"] >= START_SEASON) &
    (season_clean["season"] <= END_SEASON),
    ["season", "home_away_games", "finals_games", "overall_games"]
].copy()

validation_counts = season_summary_counts.merge(
    parsed_counts,
    on="season",
    how="left"
)

validation_counts["home_away_match"] = (
    validation_counts["home_away_games"] == validation_counts["parsed_home_away_games"]
)

validation_counts["finals_match"] = (
    validation_counts["finals_games"] == validation_counts["parsed_finals_games"]
)

validation_counts["overall_match"] = (
    validation_counts["overall_games"] == validation_counts["parsed_overall_games"]
)

validation_counts

,season,home_away_games,finals_games,overall_games,parsed_finals_games,parsed_home_away_games,parsed_overall_games,home_away_match,finals_match,overall_match
0,2012,198,9.0,207,9,198,207,True,True,True
1,2013,198,9.0,207,9,198,207,True,True,True
2,2014,198,9.0,207,9,198,207,True,True,True
3,2015,197,9.0,206,9,197,206,True,True,True
4,2016,198,9.0,207,9,198,207,True,True,True
5,2017,198,9.0,207,9,198,207,True,True,True
6,2018,198,9.0,207,9,198,207,True,True,True
7,2019,198,9.0,207,9,198,207,True,True,True
8,2020,124,9.0,133,9,124,133,True,True,True
9,2021,161,9.0,170,9,161,170,True,True,True


In [53]:
# I am adding project-specific flags to the full match-level dataset.
# These flags make it easy to filter the clean dataset later in SQL analysis.

match_all_final = match_all_clean.copy()

COVID_AFFECTED_SEASONS = [2020, 2021]

match_all_final["covid_affected"] = match_all_final["season"].isin(COVID_AFFECTED_SEASONS)

match_all_final["included_in_main_analysis"] = (
    (match_all_final["match_type"] == "Home & Away")
    & (~match_all_final["season"].isin(COVID_AFFECTED_SEASONS))
)

match_all_final[
    [
        "season",
        "round",
        "match_type",
        "home_team",
        "away_team",
        "attendance",
        "covid_affected",
        "included_in_main_analysis"
    ]
].head()

,season,round,match_type,home_team,away_team,attendance,covid_affected,included_in_main_analysis
0,2012,1,Home & Away,Greater Western Sydney,Sydney,38203,False,True
1,2012,1,Home & Away,Richmond,Carlton,78285,False,True
2,2012,1,Home & Away,Hawthorn,Collingwood,78466,False,True
3,2012,1,Home & Away,Melbourne,Brisbane Lions,33473,False,True
4,2012,1,Home & Away,Gold Coast,Adelaide,12790,False,True


In [54]:
# I am checking the final analysis flags before saving the clean dataset.

print("COVID affected counts:")
print(match_all_final["covid_affected"].value_counts())

print("\nMain analysis inclusion counts:")
print(match_all_final["included_in_main_analysis"].value_counts())

print("\nMain analysis rows by season:")
print(
    match_all_final
    .loc[match_all_final["included_in_main_analysis"] == True]
    .groupby("season")
    .size()
)

COVID affected counts:
covid_affected
False    2510
True      303
Name: count, dtype: int64

Main analysis inclusion counts:
included_in_main_analysis
True     2402
False     411
Name: count, dtype: int64

Main analysis rows by season:
season
2012    198
2013    198
2014    198
2015    197
2016    198
2017    198
2018    198
2019    198
2022    198
2023    207
2024    207
2025    207
dtype: int64


In [55]:
# I am removing the table_index debugging column and arranging columns in a cleaner order.

columns_to_keep = [
    "season",
    "year",
    "round",
    "match_type",
    "final_stage",
    "is_final",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "winner",
    "margin",
    "result_type",
    "match_day",
    "match_date",
    "month",
    "match_time",
    "attendance",
    "venue",
    "covid_affected",
    "included_in_main_analysis"
]

match_all_final = match_all_final[columns_to_keep]

match_all_final.head()

,season,year,round,match_type,final_stage,is_final,home_team,away_team,home_score,away_score,...,margin,result_type,match_day,match_date,month,match_time,attendance,venue,covid_affected,included_in_main_analysis
0,2012,2012,1,Home & Away,Not applicable,False,Greater Western Sydney,Sydney,37,100,...,63,Win,Sat,2012-03-24,March,7:20 PM,38203,Stadium Australia,False,True
1,2012,2012,1,Home & Away,Not applicable,False,Richmond,Carlton,81,125,...,44,Win,Thu,2012-03-29,March,7:45 PM,78285,M.C.G.,False,True
2,2012,2012,1,Home & Away,Not applicable,False,Hawthorn,Collingwood,137,115,...,22,Win,Fri,2012-03-30,March,7:50 PM,78466,M.C.G.,False,True
3,2012,2012,1,Home & Away,Not applicable,False,Melbourne,Brisbane Lions,78,119,...,41,Win,Sat,2012-03-31,March,1:45 PM,33473,M.C.G.,False,True
4,2012,2012,1,Home & Away,Not applicable,False,Gold Coast,Adelaide,68,137,...,69,Win,Sat,2012-03-31,March,3:45 PM,12790,Carrara,False,True


In [56]:
# I am saving the final cleaned match-level AFL attendance dataset.

match_output_path = PROCESSED_DIR / "afl_match_attendance_clean.csv"

match_all_final.to_csv(match_output_path, index=False)

print("Cleaned match-level dataset saved to:")
print(match_output_path)

Cleaned match-level dataset saved to:
C:\Users\DELL\afl-attendance-analytics\data\processed\afl_match_attendance_clean.csv


In [57]:
# I am reloading the saved match-level CSV to confirm it was saved correctly.

match_check = pd.read_csv(PROCESSED_DIR / "afl_match_attendance_clean.csv")

print("Reloaded shape:", match_check.shape)

print("\nColumns:")
print(match_check.columns.tolist())

print("\nMatch type counts:")
print(match_check["match_type"].value_counts())

print("\nMain analysis counts:")
print(match_check["included_in_main_analysis"].value_counts())

match_check.head()

Reloaded shape: (2813, 21)

Columns:
['season', 'year', 'round', 'match_type', 'final_stage', 'is_final', 'home_team', 'away_team', 'home_score', 'away_score', 'winner', 'margin', 'result_type', 'match_day', 'match_date', 'month', 'match_time', 'attendance', 'venue', 'covid_affected', 'included_in_main_analysis']

Match type counts:
match_type
Home & Away    2687
Finals          126
Name: count, dtype: int64

Main analysis counts:
included_in_main_analysis
True     2402
False     411
Name: count, dtype: int64


,season,year,round,match_type,final_stage,is_final,home_team,away_team,home_score,away_score,...,margin,result_type,match_day,match_date,month,match_time,attendance,venue,covid_affected,included_in_main_analysis
0,2012,2012,1.0,Home & Away,Not applicable,False,Greater Western Sydney,Sydney,37,100,...,63,Win,Sat,2012-03-24,March,7:20 PM,38203,Stadium Australia,False,True
1,2012,2012,1.0,Home & Away,Not applicable,False,Richmond,Carlton,81,125,...,44,Win,Thu,2012-03-29,March,7:45 PM,78285,M.C.G.,False,True
2,2012,2012,1.0,Home & Away,Not applicable,False,Hawthorn,Collingwood,137,115,...,22,Win,Fri,2012-03-30,March,7:50 PM,78466,M.C.G.,False,True
3,2012,2012,1.0,Home & Away,Not applicable,False,Melbourne,Brisbane Lions,78,119,...,41,Win,Sat,2012-03-31,March,1:45 PM,33473,M.C.G.,False,True
4,2012,2012,1.0,Home & Away,Not applicable,False,Gold Coast,Adelaide,68,137,...,69,Win,Sat,2012-03-31,March,3:45 PM,12790,Carrara,False,True
